# MeguvizCAC Turbo v2 Studio — Colab G4 (RTX PRO 6000 Blackwell 96 GB)

Structured multi-cell foreground notebook designed for **zero idle-disconnect risk** (NO `nohup` background daemons) and **maximum GPU utilization**:
- **Rust (`meguviz_native`)** Zero-GIL Rayon parallel frontend (`35x–200x` faster feature extraction, wheel cached on Drive)
- **Turbo v2 Trainer**: `torch.compile` + bucketed seq lengths + GPU `_foreach_lerp_` EMA (`54x`) + batched auxiliary passes (`7.9x`) + multi-thread prefetcher + cached OOF CharLM
- **Optional CUDA MPS Parallel Packing**: run Track A + Track B (or 2–4 seeds/ablations) **simultaneously** on one 96 GB Blackwell GPU!

In [ ]:
# Cell 1: Mount Drive, Pull Latest Code, Install/Cache Rust `meguviz_native` Wheel (~15s after first build)
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_DIR = Path('/content/MeguvizCAC')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jichuo1/MeguvizCAC.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)

subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)

# Install or build-and-cache Rust `meguviz_native` wheel to Drive
WHEEL_DIR = Path('/content/drive/MyDrive/MeguvizCAC/wheels')
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whls = sorted(WHEEL_DIR.glob(f'meguviz_native-*-{py_tag}-*x86_64.whl'), key=lambda p: p.stat().st_mtime, reverse=True)
if whls:
    print('Installing cached Rust wheel:', whls[0])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', str(whls[0])], check=True)
else:
    print('Building Rust meguviz_native wheel once & caching to Drive...')
    if subprocess.run(['bash', '-lc', 'command -v cargo'], capture_output=True).returncode != 0:
        subprocess.run(['bash', '-lc', 'curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'], check=True)
    subprocess.run([
        'bash', '-lc',
        f'export PATH="$HOME/.cargo/bin:$PATH" && maturin build --release --manifest-path {REPO_DIR}/meguviz_native/Cargo.toml -i {sys.executable} -o {WHEEL_DIR}'
    ], check=True)
    whls = sorted(WHEEL_DIR.glob(f'meguviz_native-*-{py_tag}-*linux_x86_64.whl'), key=lambda p: p.stat().st_mtime, reverse=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', str(whls[0])], check=True)

import _meguviz_rs
assert hasattr(_meguviz_rs, 'RustOOFCharLMSet')
print('Rust _meguviz_rs READY:', _meguviz_rs.__file__)

In [ ]:
# Cell 2: Stage 1 — Incremental Super-Teacher Soft-Logit Rescoring (auto-skips if input SHA matches Drive cache!)
!cd /content/MeguvizCAC && RUN_TRACK_A=0 RUN_TRACK_B=0 SKIP_RESCORE=0 bash /content/MeguvizCAC/meguviz_cac/run_v161_colab.sh

In [ ]:
# Cell 3A: Native Foreground Turbo v2 Training (Single-Stream Sequential — Zero Idle Disconnect!)
# Adjust BS=256 (or 128), USE_COMPILE=1, RUN_TRACK_A/RUN_TRACK_B as needed.
!cd /content/MeguvizCAC && SKIP_RESCORE=1 BS=256 USE_COMPILE=1 COMPILE_MODE=default PREFETCH_WORKERS=2 bash /content/MeguvizCAC/meguviz_cac/run_v161_colab.sh

In [ ]:
# Cell 3B (OPTIONAL): 96 GB Blackwell CUDA MPS Parallel Multi-Run Packing (Run 2 Tracks Simultaneously in 1 Wall-Clock Slot!)
# Prerequisites: shared_r0_v161_frozen.pt and shared_oof_charlm_v161.pkl already exist in Drive (or run Track A first).
# Keep this cell in foreground — waits for both processes while streaming live status every 30s.
import subprocess, time
from pathlib import Path

env_base = {
    **os.environ,
    'SKIP_RESCORE': '1',
    'BS': '256',
    'USE_COMPILE': '1',
    'PREFETCH_WORKERS': '2',
}
print('Starting CUDA MPS daemon for multi-process 96 GB Blackwell packing...')
subprocess.run(['bash', '-lc', 'nvidia-cuda-mps-control -d || true'])

p_a = subprocess.Popen(['bash', '/content/MeguvizCAC/meguviz_cac/run_v161_colab.sh'], env={**env_base, 'RUN_TRACK_A': '1', 'RUN_TRACK_B': '0'})
p_b = subprocess.Popen(['bash', '/content/MeguvizCAC/meguviz_cac/run_v161_colab.sh'], env={**env_base, 'RUN_TRACK_A': '0', 'RUN_TRACK_B': '1'})

t0 = time.time()
while p_a.poll() is None or p_b.poll() is None:
    el = int(time.time() - t0)
    print(f'[{el:4d}s] Track-A={"RUN" if p_a.poll() is None else p_a.returncode} | Track-B={"RUN" if p_b.poll() is None else p_b.returncode}', flush=True)
    time.sleep(30)

print('Parallel MPS finish codes:', p_a.returncode, p_b.returncode)
assert p_a.returncode == 0 and p_b.returncode == 0

In [ ]:
# Cell 4: Final Scoreboard & Drive Artifact Verification
import json
from pathlib import Path

drv = Path('/content/drive/MyDrive/MeguvizCAC/v161_production')
cmp = drv / 'v161_ab_comparison.json'
if cmp.is_file():
    print(json.dumps(json.loads(cmp.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
for sub in ('v1.6.1-A', 'v1.6.1-B'):
    d = drv / sub
    if d.is_dir():
        print(f'\n=== {d} ===')
        for f in sorted(d.iterdir()):
            if f.is_file():
                print(f'  {f.name:52s} {f.stat().st_size / 1e6:7.2f} MB')